# 05 — Exposure

WorldPop and OSM/HDX layers for the corridor. This is the raw material Phase 5
turns into lead times and preparedness profiles; here we just check it's real
and sane.

In [1]:
import sys
sys.path.insert(0, "..")
import geopandas as gpd
import rasterio
import numpy as np
from pathlib import Path

with rasterio.open("../data/bronze/worldpop/npl_ppp_2020_constrained.tif") as ds:
    pop = ds.read(1)
    pop = np.where(np.isfinite(pop) & (pop > 0), pop, 0)
    print(f"corridor grid: {ds.width}x{ds.height} @ {ds.res} deg, crs={ds.crs}")
    print(f"total modelled population in corridor bbox: {pop.sum():,.0f}")

corridor grid: 420x900 @ (0.0008333333300245613, 0.000833333329945111) deg, crs=EPSG:4326
total modelled population in corridor bbox: 170,777


**170,777 people modelled in the corridor bounding box** — order-of-magnitude,
not a count. WorldPop models usual residence in 2020 and cannot show anyone
displaced since 26 August 2026.

## HOT/OSM exposure layers (from HDX `hot_flood_npl`)

In [2]:
import glob
for p in sorted(glob.glob("../data/silver/hot_flood_npl/*.parquet")):
    g = gpd.read_parquet(p)
    name = Path(p).stem
    print(f"{name[:50]:52} {len(g):6} features  {g.geom_type.mode()[0] if len(g) else '-'}")

Area of Interest, GeoJSON                                 1 features  Polygon
Bridge Damage Assessment, GeoJSON                        59 features  Point


Bridges (OSM), GeoJSON                                  171 features  LineString
Destroyed and Damaged Features (OSM), GeoJSON          2092 features  LineString
Education Facilities (OSM), GeoJSON                      60 features  Point
Education Facilities (Overture), GeoJSON                 13 features  Point
Exposed Hydropowers, GeoJSON                             10 features  Point


Financial Services (OSM), GeoJSON                        29 features  Point
Flood Extent, Observed 27 August 2026, GeoJSON            1 features  Polygon


Health Facilities (OSM), GeoJSON                          5 features  Point
Health Facilities (Overture), GeoJSON                     3 features  Point
Helipads (OSM), GeoJSON                                  15 features  LineString
Mapping Task Boundaries (Tasking Manager), GeoJSON        8 features  MultiPolygon
Open Spaces (OSM), GeoJSON                               65 features  Polygon


Points of Interest (OSM), GeoJSON                       397 features  Point
Points of Interest (Overture), GeoJSON                  170 features  Point


Police Stations (OSM), GeoJSON                            9 features  Point
Residential Areas (OSM), GeoJSON                        527 features  Polygon
Settlement Names (OSM), GeoJSON                          60 features  Point
Waterways (OSM), GeoJSON                                380 features  LineString


## Damage assessment (Microsoft AI for Good, via HDX)

In [3]:
for p in sorted(glob.glob("../data/silver/hot_flood_npl_buildings_damage/*.parquet")):
    g = gpd.read_parquet(p)
    print(f"{Path(p).stem}: {len(g)} features")
    if len(g):
        print(g.columns.tolist())
        display_cols = [c for c in g.columns if c.lower() in
                        ("damage", "damage_type", "class", "status")]
        for c in display_cols:
            print(g[c].value_counts())

Analyzed area (AOI clipped to post imagery extent), GeoJSON: 1 features
['name', 'geometry']


Building damage assessment, GeoJSON: 1053 features
['osm_id', 'damage_class', 'damage', 'damage_confidence', 'geometry']
damage
destroyed       677
minor-damage    155
no-damage       113
major-damage    105
no-data           3
Name: count, dtype: int64


**Caveat, carried to every rendering that uses this layer:** binary damage flag
only — no severity, occupancy, or casualty information, and nothing under cloud,
tree canopy, or debris. Buildings absent from the OSM footprint layer never enter
any score. `independence_group: cv_damage_vhr` — any other computer-vision damage
layer built from the same post-event imagery counts as the same line of evidence,
not a second one.

## Hydropower exposure

In [4]:
for p in sorted(glob.glob("../data/silver/hot_flood_npl/*Hydropower*.parquet")):
    g = gpd.read_parquet(p)
    print(f"{Path(p).stem}: {len(g)} facilities")
    for col in ("name", "Name", "NAME"):
        if col in g.columns:
            print(g[col].tolist())
            break

Exposed Hydropowers, GeoJSON: 10 facilities
['Bhotekoshi Khola Hydropower Project', 'Devighat', 'Rasuwa Bhotekoshi', 'Rasuwagadhi', 'Trishuli', 'Trishuli Galchhi', 'Upper Trishuli 3A', 'Upper Trishuli 3B', 'Upper Trishuli-1', 'Upper Trishuli-I Cascade HEP']
